## Problema 4: Predicción de precios del Mercado

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [2]:
# CARGA DE DATOS
df_precios  = pd.read_csv("datos/precios_mercado.csv")

In [3]:
# HACEMOS UNA COPIA PARA NO MODIFICAR EL ORIGINAL
df_precios_copy1 = df_precios.copy()
df_precios_copy2 = df_precios.copy()

In [4]:
df_precios_copy1['fecha'] = pd.to_datetime(df_precios_copy1['fecha'])
df_precios_copy1 = df_precios_copy1.sort_values(['pais', 'producto', 'fecha'])

# OBJETIVO (3 CLASES)
df_precios_copy1['cambio_precio'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change()

def clasificar(x):
    if x > 0.003:
        return 2   # sube
    elif x < -0.003:
        return 0  # baja
    else:
        return 1   # estable

df_precios_copy1['sube_baja'] = df_precios_copy1.groupby(['pais', 'producto'])['cambio_precio'].shift(-1)
df_precios_copy1['sube_baja'] = df_precios_copy1['sube_baja'].apply(clasificar)

# FEATURES 
for i in range(1, 10):
    df_precios_copy1[f'precio_dia_{i}'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].shift(i)

df_precios_copy1['cambio_1_dia'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change(1)
df_precios_copy1['cambio_3_dias'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change(3)

df_precios_copy1['media_5_dias'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(5).mean())
df_precios_copy1['media_10_dias'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(10).mean())

df_precios_copy1['tendencia'] = df_precios_copy1['media_5_dias'] - df_precios_copy1['media_10_dias']

df_precios_copy1['variacion'] = df_precios_copy1.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(5).std())

if 'volumen_operado_ton' in df_precios_copy1.columns:
    df_precios_copy1['cambio_volumen'] = df_precios_copy1.groupby(['pais', 'producto'])['volumen_operado_ton'].pct_change()
else:
    df_precios_copy1['cambio_volumen'] = 0

# LIMPIEZA
df_precios_copy1 = df_precios_copy1.fillna(0)

# VARIABLES
columnas_modelo = [col for col in df_precios_copy1.columns if 'precio_dia_' in col] + [
    'cambio_1_dia', 'cambio_3_dias',
    'media_5_dias', 'media_10_dias',
    'tendencia', 'variacion', 'cambio_volumen'
]

# MODELO
resultados1 = []

for producto in df_precios_copy1['producto'].unique():

    df_producto = df_precios_copy1[df_precios_copy1['producto'] == producto]

    if len(df_producto) < 60:
        continue

    X = df_producto[columnas_modelo]
    y = df_producto['sube_baja']

    punto_corte = int(len(df_producto) * 0.8)

    X_train = X.iloc[:punto_corte]
    X_test = X.iloc[punto_corte:]

    y_train = y.iloc[:punto_corte]
    y_test = y.iloc[punto_corte:]

    modelo = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        objective='multi:softmax',
        num_class=3
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, predicciones)

    print(f"{producto}: {accuracy:.3f}")

    resultados1.append(accuracy)

if resultados1:
    print("\nAccuracy medio:", np.mean(resultados1))
else:
    print("No hay suficientes datos")

Arroz: 0.348
Carne_bovina: 0.458
Carne_porcina: 0.667
Leche: 0.625
Maíz: 0.464
Soja: 0.615
Trigo: 0.667

Accuracy medio: 0.5491661547562169


Prueba del modelo 2 sin parametro estable para reducir el ruido

In [5]:
# PREPROCESAMIENTO
df_precios_copy2['fecha'] = pd.to_datetime(df_precios_copy2['fecha'])
df_precios_copy2 = df_precios_copy2.sort_values(['pais', 'producto', 'fecha'])

In [6]:
# CREAR OBJETIVO (SUBE / BAJA)
df_precios_copy2['cambio_precio'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change()

df_precios_copy2['sube_baja'] = df_precios_copy2.groupby(['pais', 'producto'])['cambio_precio'].shift(-1)
df_precios_copy2['sube_baja'] = (df_precios_copy2['sube_baja'] > 0).astype(int)

In [7]:
# FEATURES 
# Precio 10 días anteriores 
for i in range(1, 10):
    df_precios_copy2[f'precio_dia_{i}'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].shift(i)

# Cambios recientes
df_precios_copy2['cambio_1_dia'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change(1)
df_precios_copy2['cambio_3_dias'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].pct_change(3)

# Media de precios
df_precios_copy2['media_5_dias'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(5).mean())
df_precios_copy2['media_10_dias'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(10).mean())

# Tendencia simple
df_precios_copy2['tendencia'] = df_precios_copy2['media_5_dias'] - df_precios_copy2['media_10_dias']

# Variación de precios
df_precios_copy2['variacion'] = df_precios_copy2.groupby(['pais', 'producto'])['precio_usd_ton'].transform(lambda x: x.rolling(5).std())

# Volumen 
if 'volumen_operado_ton' in df_precios_copy2.columns:
    df_precios_copy2['cambio_volumen'] = df_precios_copy2.groupby(['pais', 'producto'])['volumen_operado_ton'].pct_change()
else:
    df_precios_copy2['cambio_volumen'] = 0

In [8]:
# LIMPIEZA
# Rellenar valores faltantes
df_precios_copy2 = df_precios_copy2.fillna(0)

# Quitar ruido
df_precios_copy2 = df_precios_copy2[df_precios_copy2['cambio_precio'].abs() > 0.003]

In [9]:
# VARIABLES DEL MODELO
columnas_modelo = [col for col in df_precios_copy2.columns if 'precio_dia_' in col] + [
    'cambio_1_dia', 'cambio_3_dias',
    'media_5_dias', 'media_10_dias',
    'tendencia', 'variacion', 'cambio_volumen'
]

In [10]:
# ENTRENAMIENTO DEL MODELO POR PRODUCTO
resultados2 = []

for producto in df_precios_copy2['producto'].unique():

    df_producto = df_precios_copy2[df_precios_copy2['producto'] == producto]

    # Evitar productos con pocos datos
    if len(df_producto) < 60:
        continue

    X = df_producto[columnas_modelo]
    y = df_producto['sube_baja']

    # Separar en entrenamiento y test (80% - 20%)
    punto_corte = int(len(df_producto) * 0.8)

    X_train = X.iloc[:punto_corte]
    X_test = X.iloc[punto_corte:]

    y_train = y.iloc[:punto_corte]
    y_test = y.iloc[punto_corte:]

    # Modelo XGBoost
    modelo = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05
    )

    modelo.fit(X_train, y_train)

    predicciones = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, predicciones)

    print(f"{producto}: {accuracy:.3f}")

    resultados2.append(accuracy)

# RESULTADO FINAL
print("\nAccuracy medio:", np.mean(resultados2))

Arroz: 0.524
Carne_bovina: 0.714
Carne_porcina: 0.760
Leche: 0.762
Maíz: 0.760
Soja: 0.667
Trigo: 0.700

Accuracy medio: 0.6980952380952381


In [11]:
import matplotlib.pyplot as plt

# Comparar resultados de ambos modelos
print("Modelo 1 (3 clases: sube/baja/estable):")
print(f"Accuracies: {resultados1}")
print(f"Accuracy medio: {np.mean(resultados1):.4f}")
print(f"Desviación estándar: {np.std(resultados1):.4f}")
print(f"Número de productos: {len(resultados1)}")

print("\nModelo 2 (2 clases: sube/baja, sin ruido):")
print(f"Accuracies: {resultados2}")
print(f"Accuracy medio: {np.mean(resultados2):.4f}")
print(f"Desviación estándar: {np.std(resultados2):.4f}")
print(f"Número de productos: {len(resultados2)}")

print(f"\nDiferencia: {abs(np.mean(resultados2) - np.mean(resultados1)):.4f}")

Modelo 1 (3 clases: sube/baja/estable):
Accuracies: [0.34782608695652173, 0.4583333333333333, 0.6666666666666666, 0.625, 0.4642857142857143, 0.6153846153846154, 0.6666666666666666]
Accuracy medio: 0.5492
Desviación estándar: 0.1157
Número de productos: 7

Modelo 2 (2 clases: sube/baja, sin ruido):
Accuracies: [0.5238095238095238, 0.7142857142857143, 0.76, 0.7619047619047619, 0.76, 0.6666666666666666, 0.7]
Accuracy medio: 0.6981
Desviación estándar: 0.0787
Número de productos: 7

Diferencia: 0.1489
